In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter

PROJECT_ROOT  = Path('/content/drive/MyDrive/TA_SER')

RAW_DATA_DIR  = PROJECT_ROOT / 'data'
RAVDESS_DIR   = RAW_DATA_DIR / 'RAVDESS'
EMODB_DIR     = RAW_DATA_DIR / 'EMODB'
TESS_DIR      = RAW_DATA_DIR / 'TESS'
SAVEE_DIR     = RAW_DATA_DIR / 'SAVEE'

PROCESSED_DIR = PROJECT_ROOT / 'data'
OUTPUT_CSV    = PROCESSED_DIR / 'dataset_split_v4.csv'

TRAIN_RATIO   = 0.80
VAL_RATIO     = 0.10
TEST_RATIO    = 0.10
SEED          = 42

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TARGET_LABELS = ['angry', 'happy', 'neutral', 'sad']

print(f"Project root : {PROJECT_ROOT}")
print(f"Output CSV   : {OUTPUT_CSV}")
print(f"Target labels: {TARGET_LABELS}")

Mounted at /content/drive
Project root : /content/drive/MyDrive/TA_SER
Output CSV   : /content/drive/MyDrive/TA_SER/data/dataset_split_v4.csv
Target labels: ['angry', 'happy', 'neutral', 'sad']


In [2]:
RAVDESS_MAP = {
    '01': 'neutral',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
}

EMODB_MAP = {
    'W': 'angry',
    'F': 'happy',
    'N': 'neutral',
    'T': 'sad',
}

TESS_MAP = {
    'angry'   : 'angry',
    'happy'   : 'happy',
    'neutral' : 'neutral',
    'sad'     : 'sad',
}

SAVEE_MAP = {
    'a'  : 'angry',
    'h'  : 'happy',
    'n'  : 'neutral',
    'sa' : 'sad',
}

print("Label maps defined for 4 datasets.")
print(f"  RAVDESS: keep {list(RAVDESS_MAP.values())}")
print(f"  EMODB  : keep {list(EMODB_MAP.values())}")
print(f"  TESS   : keep {list(TESS_MAP.values())}")
print(f"  SAVEE  : keep {list(SAVEE_MAP.values())}")


Label maps defined for 4 datasets.
  RAVDESS: keep ['neutral', 'happy', 'sad', 'angry']
  EMODB  : keep ['angry', 'happy', 'neutral', 'sad']
  TESS   : keep ['angry', 'happy', 'neutral', 'sad']
  SAVEE  : keep ['angry', 'happy', 'neutral', 'sad']


In [3]:
def scan_ravdess(root_dir):
    rows = []
    # safety check supaya tidak error kalau folder tidak ada
    root = Path(root_dir)
    if not root.exists():
        print(f"  RAVDESS folder tidak ditemukan: {root}")
        return rows
    # recursive glob, mencari semua file .wav
    for wav in root.rglob('*.wav'):
        parts = wav.stem.split('-')
        if len(parts) < 3:
            continue
        emo_code = parts[2]
        # filter dan append ke rows
        if emo_code in RAVDESS_MAP:
            rows.append({
                'filepath': str(wav.resolve()),
                'label'   : RAVDESS_MAP[emo_code],
                'source'  : 'ravdess'
            })
    return rows

ravdess_rows = scan_ravdess(RAVDESS_DIR)
print(f"RAVDESS: {len(ravdess_rows)} files (setelah filter 4 kelas)")
if ravdess_rows:
    print(f"  Label dist: {Counter([r['label'] for r in ravdess_rows])}")


RAVDESS: 672 files (setelah filter 4 kelas)
  Label dist: Counter({'happy': 192, 'sad': 192, 'angry': 192, 'neutral': 96})


In [4]:
def scan_emodb(root_dir):
    rows = []
    # safety check supaya tidak error kalau folder tidak ada
    root = Path(root_dir)
    if not root.exists():
        print(f"  EMODB folder tidak ditemukan: {root}")
        return rows
    # recursive glob, mencari semua file .wav
    for wav in root.rglob('*.wav'):
        name = wav.stem
        if len(name) < 6:
            continue
        emo_char = name[5]
        # filter dan append ke rows
        if emo_char in EMODB_MAP:
            rows.append({
                'filepath': str(wav.resolve()),
                'label'   : EMODB_MAP[emo_char],
                'source'  : 'emodb'
            })
    return rows

emodb_rows = scan_emodb(EMODB_DIR)
print(f"EMODB: {len(emodb_rows)} files (setelah filter 4 kelas)")
if emodb_rows:
    print(f"  Label dist: {Counter([r['label'] for r in emodb_rows])}")


EMODB: 339 files (setelah filter 4 kelas)
  Label dist: Counter({'angry': 127, 'neutral': 79, 'happy': 71, 'sad': 62})


In [5]:
def scan_tess(root_dir):
    rows = []
    root = Path(root_dir)
    if not root.exists():
        print(f"  TESS folder tidak ditemukan: {root}")
        return rows

    tess_sub = root / 'TESS Toronto emotional speech set data'
    scan_root = tess_sub if tess_sub.exists() else root

    for folder in scan_root.iterdir():
        if not folder.is_dir():
            continue

        suffix = folder.name.split('_')[-1].lower()

        if suffix in TESS_MAP:
            for wav in folder.glob('*.wav'):
                rows.append({
                    'filepath': str(wav.resolve()),
                    'label'   : TESS_MAP[suffix],
                    'source'  : 'tess'
                })
    return rows

tess_rows = scan_tess(TESS_DIR)
print(f"TESS: {len(tess_rows)} files (setelah filter 4 kelas)")
if tess_rows:
    print(f"  Label dist: {Counter([r['label'] for r in tess_rows])}")

TESS: 1600 files (setelah filter 4 kelas)
  Label dist: Counter({'sad': 400, 'angry': 400, 'happy': 400, 'neutral': 400})


In [6]:
def parse_savee_prefix(stem):

    token = stem.split('_')[-1]

    if token[:2] in SAVEE_MAP:
        return token[:2]
    if token[:1] in SAVEE_MAP:
        return token[:1]
    return None

def scan_savee(root_dir):
    rows = []
    root = Path(root_dir)
    if not root.exists():
        print(f" SAVEE folder tidak ditemukan: {root}")
        return rows

    for wav in root.rglob('*.wav'):
        prefix = parse_savee_prefix(wav.stem)
        if prefix and prefix in SAVEE_MAP:
            rows.append({
                'filepath': str(wav.resolve()),
                'label'   : SAVEE_MAP[prefix],
                'source'  : 'savee'
            })
    return rows

savee_rows = scan_savee(SAVEE_DIR)
print(f"SAVEE: {len(savee_rows)} files (setelah filter 4 kelas)")
if savee_rows:
    print(f"  Label dist: {Counter([r['label'] for r in savee_rows])}")


SAVEE: 300 files (setelah filter 4 kelas)
  Label dist: Counter({'neutral': 120, 'angry': 60, 'happy': 60, 'sad': 60})


In [14]:
all_rows = ravdess_rows + emodb_rows + tess_rows + savee_rows
df = pd.DataFrame(all_rows)

print(f"Total files: {len(df)}")
print("\n--- Distribusi per source ---")
print(df['source'].value_counts())

print("\n--- Cross-tab source × label ---")
print(pd.crosstab(df['source'], df['label'], margins=True))

dup = df['filepath'].duplicated().sum()
print(f"\nDuplicate filepaths: {dup}")
assert dup == 0, "Ada duplicate filepath!"

for src in df['source'].unique():
    labels_in_src = set(df[df['source'] == src]['label'].unique())
    missing = set(TARGET_LABELS) - labels_in_src
    if missing:
        print(f" Source '{src}' missing kelas: {missing}")
    else:
        print(f" Source '{src}' lengkap 4 kelas")

sample_check = df.sample(min(5, len(df)), random_state=SEED)
print("\n--- Sanity check random 5 file ---")
for _, row in sample_check.iterrows():
    exists = Path(row['filepath']).exists()
    status = "AMAN" if exists else "TIDK"
    print(f"  {status} [{row['source']:8s}] [{row['label']:8s}] {row['filepath'][-60:]}")


Total files: 2911

--- Distribusi per source ---
source
tess       1600
ravdess     672
emodb       339
savee       300
Name: count, dtype: int64

--- Cross-tab source × label ---
label    angry  happy  neutral  sad   All
source                                   
emodb      127     71       79   62   339
ravdess    192    192       96  192   672
savee       60     60      120   60   300
tess       400    400      400  400  1600
All        779    723      695  714  2911

Duplicate filepaths: 0
 Source 'ravdess' lengkap 4 kelas
 Source 'emodb' lengkap 4 kelas
 Source 'tess' lengkap 4 kelas
 Source 'savee' lengkap 4 kelas

--- Sanity check random 5 file ---
  AMAN [ravdess ] [happy   ] yDrive/TA_SER/data/RAVDESS/Actor_07/03-01-03-02-01-02-07.wav
  AMAN [emodb   ] [neutral ] /content/drive/MyDrive/TA_SER/data/EMODB/wav/03a01Nc.wav
  AMAN [savee   ] [neutral ] /content/drive/MyDrive/TA_SER/data/SAVEE/ALL/KL_n06.wav
  AMAN [tess    ] [happy   ] onto emotional speech set data/OAF_happy/OAF_sh

In [10]:
def split_per_source(df_src, train_ratio=0.80, val_ratio=0.10, seed=42):

    train, temp = train_test_split(
        df_src,
        test_size=(1 - train_ratio),
        stratify=df_src['label'],
        random_state=seed
    )
    val, test = train_test_split(
        temp,
        test_size=0.5,
        stratify=temp['label'],
        random_state=seed
    )
    train = train.assign(split='train')
    val   = val.assign(split='val')
    test  = test.assign(split='test')
    return pd.concat([train, val, test], ignore_index=True)

splits = []

for src in df['source'].unique():
    df_src = df[df['source'] == src].reset_index(drop=True)
    df_src_split = split_per_source(df_src, TRAIN_RATIO, VAL_RATIO, SEED)
    splits.append(df_src_split)
    print(f"{src:8s}: train={len(df_src_split[df_src_split.split=='train']):4d}  "
          f"val={len(df_src_split[df_src_split.split=='val']):3d}  "
          f"test={len(df_src_split[df_src_split.split=='test']):3d}")

df_final = pd.concat(splits, ignore_index=True)
print(f"\n Total: {len(df_final)} files")


ravdess : train= 537  val= 67  test= 68
emodb   : train= 271  val= 34  test= 34
tess    : train=1280  val=160  test=160
savee   : train= 240  val= 30  test= 30

 Total: 2911 files


In [12]:
print("\n --- Distribusi final per (source, split, label) ---")
pivot = df_final.groupby(['source', 'split', 'label']).size().unstack(fill_value=0)
print(pivot)

print("\n --- Sanity check: ga ada kelas kosong di val/test per source ---")
for src in df_final['source'].unique():
    for split in ['val', 'test']:
        subset = df_final[(df_final['source']==src) & (df_final['split']==split)]
        labels_present = set(subset['label'].unique())
        missing = set(TARGET_LABELS) - labels_present
        if missing:
            print(f" {src}/{split}: MISSING {missing}")
        else:
            counts = subset['label'].value_counts().to_dict()
            print(f" {src}/{split}: {counts}")



 --- Distribusi final per (source, split, label) ---
label          angry  happy  neutral  sad
source  split                            
emodb   test      13      7        8    6
        train    101     57       63   50
        val       13      7        8    6
ravdess test      19     20        9   20
        train    154    153       77  153
        val       19     19       10   19
savee   test       6      6       12    6
        train     48     48       96   48
        val        6      6       12    6
tess    test      40     40       40   40
        train    320    320      320  320
        val       40     40       40   40

 --- Sanity check: ga ada kelas kosong di val/test per source ---
 ravdess/val: {'happy': 19, 'sad': 19, 'angry': 19, 'neutral': 10}
 ravdess/test: {'happy': 20, 'sad': 20, 'angry': 19, 'neutral': 9}
 emodb/val: {'angry': 13, 'neutral': 8, 'happy': 7, 'sad': 6}
 emodb/test: {'angry': 13, 'neutral': 8, 'happy': 7, 'sad': 6}
 tess/val: {'happy': 40, 'sad': 

In [13]:
df_final.to_csv(OUTPUT_CSV, index=False)
print(f" Saved to: {OUTPUT_CSV}")
print(f" Shape: {df_final.shape}")
print(f" Columns: {list(df_final.columns)}")
print("\n --- Head preview ---")
print(df_final.head())
print("\n --- Tail preview ---")
print(df_final.tail())


 Saved to: /content/drive/MyDrive/TA_SER/data/dataset_split_v4.csv
 Shape: (2911, 4)
 Columns: ['filepath', 'label', 'source', 'split']

 --- Head preview ---
                                            filepath  label   source  split
0  /content/drive/MyDrive/TA_SER/data/RAVDESS/Act...  angry  ravdess  train
1  /content/drive/MyDrive/TA_SER/data/RAVDESS/Act...  happy  ravdess  train
2  /content/drive/MyDrive/TA_SER/data/RAVDESS/Act...  angry  ravdess  train
3  /content/drive/MyDrive/TA_SER/data/RAVDESS/Act...    sad  ravdess  train
4  /content/drive/MyDrive/TA_SER/data/RAVDESS/Act...  angry  ravdess  train

 --- Tail preview ---
                                               filepath    label source split
2906  /content/drive/MyDrive/TA_SER/data/SAVEE/ALL/J...  neutral  savee  test
2907  /content/drive/MyDrive/TA_SER/data/SAVEE/ALL/J...  neutral  savee  test
2908  /content/drive/MyDrive/TA_SER/data/SAVEE/ALL/J...  neutral  savee  test
2909  /content/drive/MyDrive/TA_SER/data/SAVEE/ALL